# XMCA-Net-Lite: 04 - Perception Consistency Scoring (S_P)
### Purpose
Compute visual Perception Consistency Score (S_P):
1. Extract atomic claims from generated answers.
2. Detect visual regions via OWL-ViT.
3. Compute region-to-claim similarity using SigLIP/CLIP.
Outputs saved in Parquet shards.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
PROJECT_ROOT = Path("/kaggle/working/xmca") if Path("/kaggle/working").exists() else Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, set_seed
from src.perception.claims import extract_claims_from_answer
from src.perception.detector import OwlViTDetector
from src.perception.scorer import PerceptionScorer
from src.utils.checkpoints import ShardManager
from src.utils.logging_metadata import save_run_metadata

config = load_config()
seed = config["project"]["default_seed"]
set_seed(seed)


In [ ]:
# Load all Qwen2-VL generation shards
qwen_gen_dir = Path(config["paths"]["generations_dir"]) / "qwen2vl"
shard_mgr_qwen = ShardManager(qwen_gen_dir)
gen_df = shard_mgr_qwen.load_all_shards("qwen2vl_generation")
print(f"Loaded {len(gen_df)} generation items for perception evaluation.")

detector = OwlViTDetector(
    model_id=config["models"]["perception"]["detector_model_id"],
    box_threshold=config["models"]["perception"]["box_threshold"]
)
scorer = PerceptionScorer(model_id=config["models"]["perception"]["similarity_model_id"])

out_dir = Path(config["paths"]["perception_dir"])
shard_mgr_sp = ShardManager(out_dir, shard_size=config["execution"]["shard_size"])
items = gen_df.to_dict(orient="records")
remaining = shard_mgr_sp.get_remaining_items(items, id_key="id")
print(f"Remaining items to compute S_P: {len(remaining)}")

shard_records = []
shard_idx = 0

for row in remaining:
    claims = extract_claims_from_answer(row["generated_answer"])
    boxes, labels, confs = detector.detect_regions("dummy.jpg", claims)
    s_p = scorer.compute_s_p("dummy.jpg", claims, boxes)
    
    shard_records.append({
        "id": row["id"],
        "dataset": row["dataset"],
        "claims": claims,
        "boxes": boxes,
        "labels": labels,
        "s_p": s_p,
        "is_hallucination_gt": row["is_hallucination_gt"]
    })
    
    if len(shard_records) >= config["execution"]["shard_size"]:
        shard_mgr_sp.save_shard(shard_idx, shard_records, prefix="perception_scores")
        shard_records = []
        shard_idx += 1

if shard_records:
    shard_mgr_sp.save_shard(shard_idx, shard_records, prefix="perception_scores")

save_run_metadata(out_dir, config, seed, "04_perception", {"total_processed": len(items)})
print("\nStep 04 completed. Perception scores saved.")
